In [1]:
import torch

def check_cuda():
    print("🔍 Checking hardware status...")
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3) # Convert bytes to GB
        print("✅ SUCCESS: Connected to GPU!")
        print(f"🖥️  Device Name : {device_name}")
        print(f"💾 Total Memory: {gpu_memory:.2f} GB")
        return "cuda"
    else:
        print("❌ WARNING: Not connected to a GPU.")
        print("   To fix this in Colab: Go to Runtime > Change runtime type > Select T4 GPU.")
        return "cpu"

# Set the device variable for your models to use
device = check_cuda()

🔍 Checking hardware status...
✅ SUCCESS: Connected to GPU!
🖥️  Device Name : Tesla T4
💾 Total Memory: 14.56 GB


In [2]:
# ─── CELL 1: STABLE ENVIRONMENT SETUP ───

# ── Step 1: Force numpy downgrade FIRST ──
print("🔧 Fixing numpy compatibility...")
!pip install -q "numpy<2.0" --force-reinstall

# ── Step 2: Remove conflicts ──
print("🧹 Removing conflicting packages...")
!pip uninstall -y torchcodec pinecone-client > /dev/null 2>&1

# ── Step 3: sentence-transformers after numpy fix ──
print("📦 Installing sentence-transformers...")
!pip install -q sentence-transformers==2.7.0

# ── Step 4: LangChain stack ──
print("📦 Installing LangChain stack...")
!pip install -q \
    langchain \
    langchain-core \
    langchain-community \
    langchain-classic \
    langchain-groq \
    langchain-huggingface \
    langchain-pinecone

# ── Step 5: ✅ pinecone (not pinecone-client) ──
print("📦 Installing Pinecone + remaining libs...")
!pip install -q \
    pinecone \
    langgraph \
    langgraph-checkpoint-sqlite \
    pydantic \
    peft \
    bitsandbytes \
    qwen-vl-utils \
    decord \
    plotly \
    docling \
    rich \
    openai

# ── Step 6: PyTorch Optimizations ──
print("⚡ Installing PyTorch optimizations (torchao)...")
!pip install -q -U torchao

print("\n✅ All dependencies installed.")
print("⚠️  RESTART RUNTIME NOW → then run from Cell 2")

🔧 Fixing numpy compatibility...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 

In [1]:
# ─── CELL 2: MOUNT GOOGLE DRIVE & VERIFY WEIGHTS ───
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import shutil
import pandas as pd
import joblib

# 2. Define Original Drive Paths
drive_lora_path = "/content/drive/MyDrive/AI_SPORT_MODEL_WEIGHTS/LLM_MODEL_WEIGHTS"
drive_rf_dir = "/content/drive/MyDrive/AI_SPORT_MODEL_WEIGHTS/ML_MODEL_WEIGHTS"

# 3. Define Local Colab Paths (Faster Disk Access)
local_base_dir = "/content/local_models"
local_lora_path = f"{local_base_dir}/LLM_MODEL_WEIGHTS"
local_rf_dir = f"{local_base_dir}/ML_MODEL_WEIGHTS"

print("\n📦 Transferring models from Drive to Colab local memory for faster access...")

# 4. Copy Video AI LoRA Weights
if os.path.exists(drive_lora_path):
    if not os.path.exists(local_lora_path):
        shutil.copytree(drive_lora_path, local_lora_path)
    print(f"✅ Video AI (LoRA) weights copied to local memory: {local_lora_path}")
else:
    print(f"⚠️ WAIT: Could not find LoRA weights in Drive at {drive_lora_path}.")

# 5. Copy Medical AI Random Forest Model
if os.path.exists(drive_rf_dir):
    if not os.path.exists(local_rf_dir):
        shutil.copytree(drive_rf_dir, local_rf_dir)
    print(f"✅ Medical AI (Random Forest) copied to local memory: {local_rf_dir}")
else:
    print(f"⚠️ WAIT: Could not find Medical model in Drive at {drive_rf_dir}.")

# 6. Set Final Variables for the Rest of the Application
lora_path = local_lora_path
# Assumes the actual file inside the folder is named 'random_forest_model.joblib'
rf_model_path = os.path.join(local_rf_dir, "random_forest_model.joblib")

print("\nReady to load models into the system!")

# =====================================================================
# 7. LOAD DATASET AND MODEL INTO MEMORY
# =====================================================================
print("\n⏳ Loading tabular data and ML models...")

os.makedirs(local_base_dir, exist_ok=True)
shutil.copy("/content/drive/MyDrive/football_checkpoints_REAL/dataset/Workout_Routine_Dirty.csv", "/content/local_models/ML_MODEL_WEIGHTS/Workout_Routine_Dirty.csv")
# 1. Load your player dataset (UPDATE THIS PATH TO YOUR ACTUAL CSV!)
player_df = pd.read_csv("/content/local_models/ML_MODEL_WEIGHTS/Workout_Routine_Dirty.csv")

# 2. Define the exact features your Random Forest model expects
features = ['Max_Speed', 'Distance', 'Acceleration_Count', 'Sleep_Quality', 'Soreness','RPE','Stress']

# 3. Load the Random Forest model into memory using the path defined above
medical_model = joblib.load(rf_model_path)

print("✅ player_df, medical_model, and features are now successfully loaded into the Coordinator!")


📦 Transferring models from Drive to Colab local memory for faster access...
✅ Video AI (LoRA) weights copied to local memory: /content/local_models/LLM_MODEL_WEIGHTS
✅ Medical AI (Random Forest) copied to local memory: /content/local_models/ML_MODEL_WEIGHTS

Ready to load models into the system!

⏳ Loading tabular data and ML models...
✅ player_df, medical_model, and features are now successfully loaded into the Coordinator!


In [3]:
# ─── CELL 2.5: STAGE NOTEBOOKS IN LOCAL DIRECTORY ───
import os
import shutil

# 1. Where are your 5 notebooks saved in your Google Drive? (UPDATE THIS)
drive_nodes_path = "/content/drive/MyDrive/AI_SPORT_MODEL_WEIGHTS/NOTEBOOK_NODE_MODEL"

# 2. The clean, organized folder we are creating inside Colab
local_nodes_dir = "/content/system_nodes"

print("📦 Transferring sub-nodes to organized local directory...")

# 3. Copy the folder over
if os.path.exists(drive_nodes_path):
    if not os.path.exists(local_nodes_dir):
        shutil.copytree(drive_nodes_path, local_nodes_dir)
    print(f"✅ All nodes copied to local directory: {local_nodes_dir}")
else:
    print(f"⚠️ WAIT: Could not find nodes folder at {drive_nodes_path}. Please check your Drive path.")

📦 Transferring sub-nodes to organized local directory...
✅ All nodes copied to local directory: /content/system_nodes


In [4]:
import shutil

source_dir = '/content/drive/MyDrive/football_checkpoints_REAL'
destination_dir = '/content/football_checkpoints_REAL'

# copytree copies the directory and all of its contents
shutil.copytree(source_dir, destination_dir,dirs_exist_ok=True)

print("Directory copied successfully!")

Directory copied successfully!


In [ ]:
# ─── CELL 3: API KEYS ───
import os

# Set your API keys securely in the environment variables
os.environ["GROQ_API_KEY"] = "######"
os.environ["PINECONE_API_KEY"] = "######"

print("✅ API keys set.")

✅ API keys set.


In [6]:
# ─── CELL 4: LOAD ALL NODES VIA %run (UPDATED PATHS) ───

print("⏳ Loading all agent nodes from local memory...")
print("━"*60)

# ── Node 1: Medical RAG ──
print("📡 Loading RAG Medical node...")
%run /content/system_nodes/NODE_sport_AI_RAG_system.ipynb
print("✅ medical_rag_chain ready\n")

# ── Node 2: Reviewer ──
print("🔍 Loading Reviewer node...")
%run /content/system_nodes/NODE_reviewer_for_RAG.ipynb
print("✅ reviewer_chain + ask_with_review() ready\n")

# ── Node 3: Guide ──
print("🧭 Loading Guide node...")
%run /content/system_nodes/NODE_USER_Guide_RAG_SPORT.ipynb
print("✅ guide_chain + ask_guide() ready\n")

# ── Node 4: Profile/CV Feedback ──
print("📄 Loading Profile Analyst node...")
%run /content/system_nodes/NODE_Profile_Analyst_feedback.ipynb
print("✅ analyze_athletic_profile() ready\n")

# ── Node 5: Video Analysis & Valuation ──
print("🎥 Loading Video Analysis node...")
%run /content/system_nodes/NODE_Video_Analysis_Scout_Vision.ipynb
print("✅ analyze_video_chunk() + calculate_market_value() ready\n")

print("━"*60)
print("✅ ALL 5 NODES LOADED INTO KERNEL MEMORY")

⏳ Loading all agent nodes from local memory...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📡 Loading RAG Medical node...
✅ Medical/Sport RAG libraries imported successfully.
✅ Medical/Sport RAG libraries imported successfully.
🔌 Loading embedding model and connecting to Pinecone namespaces...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

✅ analyze_video_chunk() + calculate_market_value() ready

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ ALL 5 NODES LOADED INTO KERNEL MEMORY


In [7]:
# ─── CELL 5: LANGGRAPH IMPORTS ───
from typing import TypedDict, Annotated, Literal
from operator import add
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

print("✅ LangGraph imports ready.")

✅ LangGraph imports ready.


In [8]:
# ─── CELL 6: SHARED GRAPH STATE ───
class GraphState(TypedDict):
    # ── Input ──────────────────────────────────
    user_input:           str
    file_path:            str
    has_file:             bool

    # ── Memory ─────────────────────────────────
    chat_history:         Annotated[list[BaseMessage], add]

    # ── Routing ────────────────────────────────
    route:                str

    # ── RAG + Reviewer ─────────────────────────
    rag_answer:           str
    rag_context:          list
    reviewer_verdict:     str
    reviewer_output:      dict

    # ── Other nodes ────────────────────────────
    guide_answer:         str
    feedback_output:      str
    player_output:        dict

    # ── Final ──────────────────────────────────
    final_answer:         str
    error:                str

print("✅ GraphState defined.")

✅ GraphState defined.


In [9]:
# ─── CELL 7: SUPERVISOR LLM + PROMPT ───
supervisor_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    max_tokens=20,
)

supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are the Supervisor of SportsMed AI — a multi-agent sports platform.
Read the user message and route to exactly ONE agent.

CONVERSATION HISTORY:
{chat_history}

AGENTS:
  medical_rag     → sports medicine, exercises, muscles, rehab,
                    nutrition, physiology, any fitness question
  guide           → how to use the app, app features,
                    what app can/cannot do, Q&A about the system
  feedback        → user uploads CV/PDF for athletic profile analysis
  player_analysis → video analysis, player card, spider diagram,
                    market value, FIFA stats, scouting report

RULES:
  • CV/PDF/portfolio upload      → feedback
  • Video/player stats request   → player_analysis
  • App usage questions          → guide
  • Medical/fitness questions    → medical_rag
  • Follow-up question           → same agent as previous turn
  • Unclear                      → medical_rag

RESPOND WITH ONLY ONE WORD:
  medical_rag  OR  guide  OR  feedback  OR  player_analysis
"""),
    ("human",
     "User message  : {user_input}\n"
     "File uploaded : {has_file}\n"
     "Route to:"
    ),
])

print("✅ Supervisor LLM + prompt ready.")

✅ Supervisor LLM + prompt ready.


In [10]:
# ─── CELL 8: SUPERVISOR NODE ───
def format_history(messages: list, max_exchanges: int = 3) -> str:
    if not messages:
        return "No previous conversation."
    recent = messages[-(max_exchanges * 2):]
    lines  = []
    for msg in recent:
        role = "User" if isinstance(msg, HumanMessage) else "Assistant"
        lines.append(f"{role}: {msg.content[:200]}")
    return "\n".join(lines)


def supervisor_node(state: GraphState) -> dict:
    print(f"\n{'━'*60}")
    print(f"🎯 SUPERVISOR")
    print(f"   Input    : {state['user_input']}")
    print(f"   Has file : {state.get('has_file', False)}")
    print(f"   History  : {len(state.get('chat_history', []))} messages")

    chain  = supervisor_prompt | supervisor_llm
    result = chain.invoke({
        "user_input"   : state["user_input"],
        "has_file"     : state.get("has_file", False),
        "chat_history" : format_history(state.get("chat_history", [])),
    })

    route = result.content.strip().lower()
    valid = ["medical_rag", "guide", "feedback", "player_analysis"]
    if route not in valid:
        print(f"   ⚠️  Invalid route '{route}' → fallback: medical_rag")
        route = "medical_rag"

    print(f"   ✅ Routed → {route.upper()}")
    print(f"{'━'*60}")

    # FIX: Removed **state
    return {
        "route"       : route,
        "chat_history": [HumanMessage(content=state["user_input"])],
    }

print("✅ Supervisor node defined.")

✅ Supervisor node defined.


In [11]:
# ─── CELL 9: ROUTER EDGE ───
def route_to_agent(
    state: GraphState,
) -> Literal["medical_rag", "guide", "feedback", "player_analysis"]:
    route = state.get("route", "medical_rag")
    print(f"   🔀 Router → {route}")
    return route

print("✅ Router edge defined.")

✅ Router edge defined.


In [12]:
# ─── CELL 10: MEDICAL RAG NODE ───
def medical_rag_node(state: GraphState) -> dict:
    print(f"\n⚕️  MEDICAL RAG NODE")
    print(f"   Question: {state['user_input']}")

    try:
        result      = medical_rag_chain.invoke({"input": state["user_input"]})
        answer      = result.get("answer", "")
        context     = result.get("context", [])
        print(f"   ✅ RAG answered — {len(context)} chunks retrieved")

        # FIX: Removed **state
        return {"rag_answer": answer, "rag_context": context}

    except Exception as e:
        err = f"RAG node error: {str(e)}"
        print(f"   ❌ {err}")
        return {"rag_answer": "", "rag_context": [], "error": err}

print("✅ Medical RAG node defined.")

✅ Medical RAG node defined.


In [13]:
# ─── CELL 11: REVIEWER NODE ───
def reviewer_node(state: GraphState) -> dict:
    print(f"\n🔍 REVIEWER NODE")

    try:
        review  = review_answer(
            question     = state["user_input"],
            rag_answer   = state["rag_answer"],
            context_docs = state["rag_context"],
        )
        verdict = review.get("verdict", "APPROVE")

        if verdict == "REVISE":
            final = review.get("revised_answer", state["rag_answer"])
            print(f"   ⚠️  REVISED — score: {review.get('score')}/10")
        elif verdict == "REJECT":
            final = ("❌ Answer rejected by quality reviewer.\n"
                     "Please rephrase your question.")
            print(f"   ❌ REJECTED")
        else:
            final = state["rag_answer"]
            print(f"   ✅ APPROVED — score: {review.get('score')}/10")

        # FIX: Removed **state
        return {
            "reviewer_verdict": verdict,
            "reviewer_output" : review,
            "final_answer"    : final,
            "chat_history"    : [AIMessage(content=final)],
        }

    except Exception as e:
        err = f"Reviewer error: {str(e)}"
        print(f"   ❌ {err}")
        return {
            "reviewer_verdict": "APPROVE",
            "final_answer"    : state["rag_answer"],
            "chat_history"    : [AIMessage(content=state["rag_answer"])],
            "error"           : err,
        }

print("✅ Reviewer node defined.")

✅ Reviewer node defined.


In [14]:
# ─── CELL 12: GUIDE NODE ───
def guide_node(state: GraphState) -> dict:
    print(f"\n🧭 GUIDE NODE")
    print(f"   Question: {state['user_input']}")

    try:
        answer = guide_chain.invoke({"question": state["user_input"]})
        print(f"   ✅ Guide answered")

        # FIX: Removed **state
        return {
            "guide_answer": answer,
            "final_answer": answer,
            "chat_history": [AIMessage(content=answer)],
        }

    except Exception as e:
        err = f"Guide node error: {str(e)}"
        print(f"   ❌ {err}")
        return {"guide_answer": "", "final_answer": "", "error": err}

print("✅ Guide node defined.")

✅ Guide node defined.


In [15]:
# ─── CELL 13: FEEDBACK / CV NODE ───
def feedback_node(state: GraphState) -> dict:
    print(f"\n📄 FEEDBACK / CV ANALYSIS NODE")

    try:
        file_path = state.get("file_path", "")
        question  = state.get("user_input", "")

        if not file_path:
            answer = ("❌ No file provided. "
                      "Please upload your CV or athletic profile PDF.")
        else:
            print(f"   📎 Analyzing: {file_path}")
            answer = analyze_athletic_profile(
                file_path         = file_path,
                specific_question = question,
            )
            print(f"   ✅ Profile analysis complete")

        # FIX: Removed **state
        return {
            "feedback_output": answer,
            "final_answer"   : answer,
            "chat_history"   : [AIMessage(content=answer)],
        }

    except Exception as e:
        err = f"Feedback node error: {str(e)}"
        print(f"   ❌ {err}")
        return {"feedback_output": "", "final_answer": "", "error": err}

print("✅ Feedback node defined.")

✅ Feedback node defined.


In [16]:
# ─── CELL 14: PLAYER ANALYSIS NODE ───
def player_analysis_node(state: GraphState) -> dict:
    print(f"\n🎮 PLAYER ANALYSIS NODE")

    try:
        file_path    = state.get("file_path", "")
        user_input   = state.get("user_input", "")

        player_index = 0
        import re
        match = re.search(r'\d+', user_input)
        if match:
            player_index = int(match.group())

        print(f"   📊 Generating player card — index: {player_index}")

        card = generate_player_card(
            player_index  = player_index,
            player_df     = player_df,
            medical_model = medical_model,
            features      = features,
        )

        video_score = 7.0
        if file_path and file_path.endswith(('.mp4', '.avi')):
            print(f"   🎬 Video detected — analyzing: {file_path}")
            action, video_score = analyze_video_chunk(file_path)
            print(f"   ✅ Video action: {action} | Score: {video_score}")

        health_score     = card['Medical_Report']['Health_Score']
        min_val, max_val = calculate_market_value(video_score, health_score)

        spider_json = visualize_player_profile(
            player_index  = player_index,
            video_score   = video_score,
            player_df     = player_df,
            medical_model = medical_model,
            features      = features,
        )

        report_prompt = generate_scouting_report_prompt(
            player_index           = player_index,
            video_performance_score= video_score,
            player_df              = player_df,
            medical_model          = medical_model,
            features               = features,
        )

        from groq import Groq
        client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
        response = client.chat.completions.create(
            model    = "llama-3.3-70b-versatile",
            messages = [{"role": "user", "content": report_prompt}],
            max_tokens = 1000,
        )
        scouting_report = response.choices[0].message.content

        answer = f"""
## 🏟️ Player Analysis Report

**Player** : {card['Name']}
**Position** : {card['Position']}

### ⚽ FIFA Stats
- PAC (Pace)     : {card['FIFA_Stats']['PAC']}
- STA (Stamina)  : {card['FIFA_Stats']['STA']}
- PHY (Physical) : {card['FIFA_Stats']['PHY']}

### 🏥 Medical Report
- Health Score   : {health_score}/100
- Status         : {card['Medical_Report']['Status']}
- Sleep Quality  : {card['Medical_Report']['Sleep_Quality']}
- Soreness Level : {card['Medical_Report']['Soreness_Level']}

### 💰 Market Valuation
- Estimated Range: ${min_val:,.0f} — ${max_val:,.0f}

### 📋 Scouting Report
{scouting_report}
"""

        print(f"   ✅ Player analysis complete")

        # FIX: Removed **state
        return {
            "player_output": spider_json,
            "final_answer" : answer,
            "chat_history" : [AIMessage(content=answer)],
        }

    except Exception as e:
        err = f"Player analysis error: {str(e)}"
        print(f"   ❌ {err}")
        return {"player_output": {}, "final_answer": "", "error": err}

print("✅ Player analysis node defined.")

✅ Player analysis node defined.


In [17]:
# ─── CELL 15: BUILD GRAPH ───
from langgraph.checkpoint.memory import MemorySaver

print("🔨 Building LangGraph multi-agent graph...")

# ✅ MemorySaver — always fresh per session
# No accumulation, no db file, no persistence issues
memory = MemorySaver()

builder = StateGraph(GraphState)

# ── Add all nodes ──
builder.add_node("supervisor",      supervisor_node)
builder.add_node("medical_rag",     medical_rag_node)
builder.add_node("reviewer",        reviewer_node)
builder.add_node("guide",           guide_node)
builder.add_node("feedback",        feedback_node)
builder.add_node("player_analysis", player_analysis_node)

# ── Entry point ──
builder.set_entry_point("supervisor")

# ── Supervisor routes ──
builder.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {
        "medical_rag"     : "medical_rag",
        "guide"           : "guide",
        "feedback"        : "feedback",
        "player_analysis" : "player_analysis",
    }
)

# ── Medical RAG → Reviewer → END ──
builder.add_edge("medical_rag", "reviewer")
builder.add_edge("reviewer",    END)

# ── All others → END directly ──
builder.add_edge("guide",           END)
builder.add_edge("feedback",        END)
builder.add_edge("player_analysis", END)

graph = builder.compile(checkpointer=memory)

print("✅ Graph compiled with MemorySaver.")
print("""
  START → supervisor
    ├→ medical_rag → reviewer → END
    ├→ guide                 → END
    ├→ feedback              → END
    └→ player_analysis       → END
""")

🔨 Building LangGraph multi-agent graph...
✅ Graph compiled with MemorySaver.

  START → supervisor
    ├→ medical_rag → reviewer → END
    ├→ guide                 → END
    ├→ feedback              → END
    └→ player_analysis       → END



In [18]:
# ─── CELL 16: CHAT FUNCTION (FIXED MEMORY) ───

MAX_HISTORY = 20  # ✅ trim memory after 20 messages

def chat(
    user_input : str,
    thread_id  : str  = "user-1",
    file_path  : str  = "",
    has_file   : bool = False,
) -> dict:

    config = {"configurable": {"thread_id": thread_id}}

    # ── Trim history if too long before invoking ──
    try:
        current_state = graph.get_state(config)
        history = current_state.values.get("chat_history", [])
        if len(history) > MAX_HISTORY:
            print(f"   🧹 Trimming memory: {len(history)} → {MAX_HISTORY} messages")
    except:
        pass

    # 🔴 FIX: ONLY pass the new inputs!
    # By removing the empty dictionaries, LangGraph will securely remember
    # the player_output from previous turns.
    result = graph.invoke(
        {
            "user_input" : user_input,
            "file_path"  : file_path,
            "has_file"   : has_file,
        },
        config=config,
    )

    # ── Build clean JSON for backend ──
    backend_response = {
        "status"      : "success" if not result.get("error") else "error",
        "thread_id"   : thread_id,
        "route"       : result.get("route", ""),
        "final_answer": result.get("final_answer", ""),
        "reviewer"    : {
            "verdict" : result.get("reviewer_verdict") or None,
            "score"   : result.get("reviewer_output", {}).get("score") or None,
        },
        "player_data" : result.get("player_output") or None,
        "error"       : result.get("error") or None,
    }

    # ── Console log ──
    print(f"\n{'━'*60}")
    print(f"📤 ROUTE   : {backend_response['route'].upper()}")
    print(f"📤 STATUS  : {backend_response['status']}")
    verdict = backend_response['reviewer']['verdict']
    score   = backend_response['reviewer']['score']
    print(f"📤 VERDICT : {verdict or '—'}  SCORE: {score or '—'}")
    print(f"{'━'*60}")
    print(result["final_answer"])
    print(f"{'━'*60}\n")

    return backend_response  # ← backend uses this dict

print("✅ chat() ready — Memory wipe bug fixed.")
print(f"   Max history: {MAX_HISTORY} messages per thread")

✅ chat() ready — Memory wipe bug fixed.
   Max history: 20 messages per thread


In [19]:
# ─── CELL 17: MEMORY INSPECTOR ───
def show_memory(thread_id: str = "user-1"):
    config  = {"configurable": {"thread_id": thread_id}}
    state   = graph.get_state(config)
    history = state.values.get("chat_history", [])

    print(f"\n{'━'*60}")
    print(f"🧠 MEMORY — Thread: {thread_id}")
    print(f"   Messages: {len(history)}")
    print(f"{'━'*60}")
    for i, msg in enumerate(history):
        role = "👤 User" if isinstance(msg, HumanMessage) else "🤖 AI"
        print(f"\n[{i+1}] {role}:")
        print(f"     {msg.content[:300]}")

print("✅ show_memory() ready.")

✅ show_memory() ready.


In [20]:
# ─── CELL 18: FULL SYSTEM TEST ───
from datetime import datetime

# ✅ Recompile graph with fresh memory before testing
memory = MemorySaver()
graph  = builder.compile(checkpointer=memory)
print("🔄 Graph recompiled with fresh memory")

THREAD = f"session-{datetime.now().strftime('%H%M%S%f')}"
print(f"🧵 Fresh Thread: {THREAD}")

# ── Test 1 ──
chat("Which muscles does the squat target?", thread_id=THREAD)

# ── Test 2 ──
chat("What can this app help me with?", thread_id=THREAD)

# # ── Test 3 ──
# chat(
#     "Please analyze my athletic profile",
#     thread_id = THREAD,
#     file_path = "/content/player_cv.pdf",
#     has_file  = True,
# )

# ── Test 4: Video Analysis Test ──
chat(
    user_input="Please analyze this football video and give me the player's performance.",
    thread_id=THREAD,
    file_path="/content/drive/MyDrive/WhatsApp Video 2026-04-30 at 4.18.34 PM.mp4",
    has_file=True
)

# ── Test 5 ──
chat("Can you tell me more about that?", thread_id=THREAD)

CURRENT_THREAD = THREAD
print(f"\n✅ Done — Thread: {CURRENT_THREAD}")

🔄 Graph recompiled with fresh memory
🧵 Fresh Thread: session-210311423116

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 SUPERVISOR
   Input    : Which muscles does the squat target?
   Has file : False
   History  : 0 messages
   ✅ Routed → MEDICAL_RAG
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   🔀 Router → medical_rag

⚕️  MEDICAL RAG NODE
   Question: Which muscles does the squat target?
   ✅ RAG answered — 6 chunks retrieved

🔍 REVIEWER NODE
🔍 Reviewer agent evaluating answer...
   ✅ APPROVED — score: 9/10

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📤 ROUTE   : MEDICAL_RAG
📤 STATUS  : success
📤 VERDICT : APPROVE  SCORE: 9
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
## 🏷️ Muscles Targeted by the Back Squat

### Overview
The back squat primarily loads the hip‑extensor and knee‑extensor muscle groups, with additional demand on core stabilizers to maintain an upright torso.

### Key Details
- **Primary Movers**
  - 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
qwen-vl-utils using decord to read video.


   🎬 Video detected — analyzing: /content/drive/MyDrive/WhatsApp Video 2026-04-30 at 4.18.34 PM.mp4


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   ✅ Video action: score | Score: 8.0
📊 Spider JSON ready for backend → Bill


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


   ✅ Player analysis complete

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📤 ROUTE   : PLAYER_ANALYSIS
📤 STATUS  : success
📤 VERDICT : APPROVE  SCORE: 9
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 🏟️ Player Analysis Report

**Player** : Bill
**Position** : Outside Back

### ⚽ FIFA Stats
- PAC (Pace)     : 98
- STA (Stamina)  : 66
- PHY (Physical) : 89

### 🏥 Medical Report
- Health Score   : 87/100
- Status         : Fit to Play
- Sleep Quality  : 6.0
- Soreness Level : 6.5

### 💰 Market Valuation
- Estimated Range: $15,030,000 — $18,370,000

### 📋 Scouting Report
**CONFIDENTIAL SCOUTING REPORT RECOMMENDATION**

To: Club Director

I am writing to provide my expert recommendation regarding the potential acquisition of Bill, an Outside Back. After carefully analyzing the data from our AI analysis results, I believe that Bill could be a valuable addition to our team.

The video analysis results indicate that Bill has a strong technical ability, with a

In [21]:
# ─── CELL 19: SESSION REPORT ───
def session_report(thread_id: str = "session-1"):
    config  = {"configurable": {"thread_id": thread_id}}
    state   = graph.get_state(config)
    history = state.values.get("chat_history", [])

    user_msgs = [m for m in history if isinstance(m, HumanMessage)]
    ai_msgs   = [m for m in history if isinstance(m, AIMessage)]

    print(f"\n{'━'*60}")
    print(f"📊 SESSION REPORT — Thread: {thread_id}")
    print(f"{'━'*60}")
    print(f"   Total messages    : {len(history)}")
    print(f"   User messages     : {len(user_msgs)}")
    print(f"   Assistant replies : {len(ai_msgs)}")
    print(f"\n   Last verdict      : {state.values.get('reviewer_verdict') or '—'}")
    print(f"   Last route        : {state.values.get('route', '—').upper()}")
    print(f"{'━'*60}")

    show_memory(thread_id)

# ✅ Uses CURRENT_THREAD from Cell 18
session_report(CURRENT_THREAD)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 SESSION REPORT — Thread: session-210311423116
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Total messages    : 8
   User messages     : 4
   Assistant replies : 4

   Last verdict      : APPROVE
   Last route        : GUIDE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🧠 MEMORY — Thread: session-210311423116
   Messages: 8
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[1] 👤 User:
     Which muscles does the squat target?

[2] 🤖 AI:
     ## 🏷️ Muscles Targeted by the Back Squat

### Overview
The back squat primarily loads the hip‑extensor and knee‑extensor muscle groups, with additional demand on core stabilizers to maintain an upright torso.

### Key Details
- **Primary Movers**
  - **Hip extensors** – gluteus maximus and hamstring

[3] 👤 User:
     What can this app help me with?

[4] 🤖 AI:
     ## 🏆 Introductio

In [22]:
# ─── CELL 20: SESSION EXPORT PIPELINE (FIXED) ───
import json
import os
from datetime import datetime
from langchain_core.messages import HumanMessage, AIMessage

def export_session(
    thread_id : str = "session-1",
    save_dir  : str = "/content/backend_outputs"
):
    os.makedirs(save_dir, exist_ok=True)

    config  = {"configurable": {"thread_id": thread_id}}
    state   = graph.get_state(config)
    history = state.values.get("chat_history", [])

    # ✅ Fixed pairing — only pair Human→AI strictly
    turns = []
    turn_number = 1
    i = 0

    while i < len(history):
        msg = history[i]

        # ✅ Only start a turn on HumanMessage
        if isinstance(msg, HumanMessage):
            user_text = msg.content

            # ✅ Look ahead for the next AIMessage
            ai_text = ""
            j = i + 1
            while j < len(history):
                if isinstance(history[j], AIMessage):
                    ai_text = history[j].content
                    i = j + 1  # skip past this AI message
                    break
                elif isinstance(history[j], HumanMessage):
                    # another human message before AI = no response
                    i = j
                    break
                j += 1
            else:
                i += 1

            # ✅ Only add if we have a real AI response
            if ai_text:
                turns.append({
                    "turn"      : turn_number,
                    "user"      : user_text,
                    "assistant" : ai_text,
                })
                turn_number += 1
        else:
            i += 1

    # ── Get last state metadata ──
    last_route   = state.values.get("route", "")
    last_verdict = state.values.get("reviewer_verdict", "")
    last_score   = state.values.get("reviewer_output", {}).get("score")
    player_data  = state.values.get("player_output") or None
    last_error   = state.values.get("error") or None

    payload = {
        "session": {
            "thread_id"      : thread_id,
            "exported_at"    : datetime.now().isoformat(),
            "total_turns"    : len(turns),
            "total_messages" : len(history),
        },
        "summary": {
            "last_route"     : last_route,
            "last_verdict"   : last_verdict or None,
            "last_score"     : last_score   or None,
            "had_error"      : bool(last_error),
        },
        "conversation": [
            {
                "turn"      : t["turn"],
                "user"      : t["user"],
                "assistant" : t["assistant"],
            }
            for t in turns
        ],
        "player_data" : player_data,
        "error"       : last_error,
    }

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename  = f"{save_dir}/session_{thread_id}_{timestamp}.json"

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    print(f"\n{'━'*60}")
    print(f"💾 SESSION EXPORTED")
    print(f"{'━'*60}")
    print(f"   Thread       : {thread_id}")
    print(f"   Raw messages : {len(history)}")
    print(f"   Clean turns  : {len(turns)}")
    print(f"   Last route   : {last_route.upper() or '—'}")
    print(f"   Last verdict : {last_verdict or '—'}")
    print(f"   Player data  : {'✅' if player_data else '—'}")
    print(f"{'━'*60}")
    print(f"📁 Saved → {filename}\n")

    return filename, payload

filename, payload = export_session(thread_id=CURRENT_THREAD)

# ── Preview the conversation structure ──
print("📋 CONVERSATION PREVIEW:")
print("━"*60)
for turn in payload["conversation"]:
    print(f"\n[Turn {turn['turn']}]")
    print(f"  👤 User      : {turn['user'][:80]}...")
    print(f"  🤖 Assistant : {turn['assistant'][:80]}...")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💾 SESSION EXPORTED
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Thread       : session-210311423116
   Raw messages : 8
   Clean turns  : 4
   Last route   : GUIDE
   Last verdict : APPROVE
   Player data  : ✅
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📁 Saved → /content/backend_outputs/session_session-210311423116_20260430_210536.json

📋 CONVERSATION PREVIEW:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Turn 1]
  👤 User      : Which muscles does the squat target?...
  🤖 Assistant : ## 🏷️ Muscles Targeted by the Back Squat

### Overview
The back squat primarily ...

[Turn 2]
  👤 User      : What can this app help me with?...
  🤖 Assistant : ## 🏆 Introduction to SportsMed AI

### Answer
The SportsMed AI app is designed t...

[Turn 3]
  👤 User      : Please analyze this football video and give me the player's performance....
  🤖 Assistant : 
## 🏟️ Player Analysis Report

*

In [23]:
# ─── CELL 21: ENHANCED JSON PREVIEW FOR BACKEND ───
import json
import os
from datetime import datetime

def preview_session_json(
    save_dir  : str = "/content/backend_outputs",
    thread_id : str = "session-1"
):
    """
    Finds the latest saved session JSON and displays
    a full enhanced preview of every section.
    """

    # ── Step 1: Find latest file for this thread ──
    files = [
        f for f in os.listdir(save_dir)
        if f.startswith(f"session_{thread_id}") and f.endswith(".json")
    ]

    if not files:
        print(f"❌ No JSON files found for thread: {thread_id}")
        return

    latest   = sorted(files)[-1]
    filepath = os.path.join(save_dir, latest)

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n{'🟦'*30}")
    print(f"📁 FILE     : {latest}")
    print(f"📍 PATH     : {filepath}")
    print(f"{'🟦'*30}")

    # ══════════════════════════════════════════
    # SECTION 1 — SESSION METADATA
    # ══════════════════════════════════════════
    session = data.get("session", {})
    print(f"""
╔══════════════════════════════════════════════╗
║           📋 SESSION METADATA                ║
╚══════════════════════════════════════════════╝
   Thread ID      : {session.get('thread_id')}
   Exported At    : {session.get('exported_at')}
   Total Turns    : {session.get('total_turns')}
   Total Messages : {session.get('total_messages')}
""")

    # ══════════════════════════════════════════
    # SECTION 2 — SESSION SUMMARY
    # ══════════════════════════════════════════
    summary = data.get("summary", {})
    verdict = summary.get("last_verdict") or "—"
    score   = summary.get("last_score")   or "—"
    print(f"""╔══════════════════════════════════════════════╗
║           📊 SESSION SUMMARY                 ║
╚══════════════════════════════════════════════╝
   Last Route     : {summary.get('last_route','—').upper()}
   Last Verdict   : {verdict}
   Last Score     : {score}
   Had Error      : {summary.get('had_error', False)}
""")

    # ══════════════════════════════════════════
    # SECTION 3 — CONVERSATION TURNS
    # ══════════════════════════════════════════
    conversation = data.get("conversation", [])
    print(f"""╔══════════════════════════════════════════════╗
║        💬 CONVERSATION ({len(conversation)} TURNS)           ║
╚══════════════════════════════════════════════╝""")

    for turn in conversation:
        print(f"""
   ┌─ Turn {turn['turn']} {'─'*40}
   │
   │  👤 USER:
   │     {turn['user'][:120]}
   │
   │  🤖 ASSISTANT (first 200 chars):
   │     {turn['assistant'][:200].replace(chr(10), ' ')}...
   │
   └{'─'*46}""")

    # ══════════════════════════════════════════
    # SECTION 4 — PLAYER / SPIDER DATA
    # ══════════════════════════════════════════
    player_data = data.get("player_data")

    if player_data:
        fifa   = player_data.get("fifa_stats", {})
        med    = player_data.get("medical_report", {})
        cats   = player_data.get("categories", [])
        scores = player_data.get("scores", [])

        print(f"""
╔══════════════════════════════════════════════╗
║        🏟️  PLAYER / SPIDER DIAGRAM DATA      ║
╚══════════════════════════════════════════════╝
   Player Name    : {player_data.get('player_name', '—')}
   Position       : {player_data.get('position', '—')}

   ⚽ FIFA Stats:
      PAC (Pace)     : {fifa.get('PAC', '—')}
      STA (Stamina)  : {fifa.get('STA', '—')}
      PHY (Physical) : {fifa.get('PHY', '—')}

   🏥 Medical Report:
      Health Score   : {med.get('Health_Score', '—')}/100
      Status         : {med.get('Status', '—')}
      Sleep Quality  : {med.get('Sleep_Quality', '—')}
      Soreness Level : {med.get('Soreness_Level', '—')}

   📊 Spider Diagram Data (backend renders this):
      Categories : {cats}
      Scores     : {scores}
""")

        # ── Visual spider bar preview ──
        print("   📊 VISUAL PREVIEW:")
        print("   " + "─"*44)
        for cat, sc in zip(cats, scores):
            bar   = "█" * int(sc / 5)
            space = "░" * (20 - int(sc / 5))
            print(f"   {cat:<12} │{bar}{space}│ {sc}")
        print("   " + "─"*44)

    else:
        print("""
╔══════════════════════════════════════════════╗
║        🏟️  PLAYER DATA                       ║
╚══════════════════════════════════════════════╝
   No player analysis in this session.
""")

    # ══════════════════════════════════════════
    # SECTION 5 — ERROR CHECK
    # ══════════════════════════════════════════
    error = data.get("error")
    print(f"""╔══════════════════════════════════════════════╗
║           ⚠️  ERROR STATUS                   ║
╚══════════════════════════════════════════════╝
   Error          : {error or '✅ None'}
""")

    # ══════════════════════════════════════════
    # SECTION 6 — RAW JSON SAMPLE FOR BACKEND
    # ══════════════════════════════════════════
    print(f"""╔══════════════════════════════════════════════╗
║     🔌 RAW JSON SAMPLE (share with backend)  ║
╚══════════════════════════════════════════════╝""")

    # Show compact version of first turn only
    sample = {
        "session"     : data["session"],
        "summary"     : data["summary"],
        "conversation": [data["conversation"][0]] if data["conversation"] else [],
        "player_data" : data["player_data"],
        "error"       : data["error"],
    }
    print(json.dumps(sample, indent=2, ensure_ascii=False)[:1500])
    print("\n   ... (full file has all turns)")

    print(f"\n{'🟦'*30}")
    print(f"✅ PREVIEW COMPLETE")
    print(f"📁 Full file at: {filepath}")
    print(f"{'🟦'*30}\n")

    return data


# ── Run preview ──
data = preview_session_json(
    save_dir  = "/content/backend_outputs",
    thread_id = CURRENT_THREAD   # ← uses fresh thread
)


🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
📁 FILE     : session_session-210311423116_20260430_210536.json
📍 PATH     : /content/backend_outputs/session_session-210311423116_20260430_210536.json
🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦

╔══════════════════════════════════════════════╗
║           📋 SESSION METADATA                ║
╚══════════════════════════════════════════════╝
   Thread ID      : session-210311423116
   Exported At    : 2026-04-30T21:05:36.037065
   Total Turns    : 4
   Total Messages : 8

╔══════════════════════════════════════════════╗
║           📊 SESSION SUMMARY                 ║
╚══════════════════════════════════════════════╝
   Last Route     : GUIDE
   Last Verdict   : APPROVE
   Last Score     : 9
   Had Error      : False

╔══════════════════════════════════════════════╗
║        💬 CONVERSATION (4 TURNS)           ║
╚══════════════════════════════════════════════╝

   ┌─ Turn 1 ────────────────────────────────────────
   │
   │  👤 USER:
   │     Which muscles does the squat t

In [24]:
# ─── CELL 22: ENHANCED SPIDER DIAGRAM VISUALIZER ───
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def display_spider_diagram_from_memory(thread_id: str):
    """
    Retrieves player_output from LangGraph memory
    and renders an enhanced Plotly spider + stats card.
    """
    try:
        # ── Step 1: Get state from memory ──
        config     = {"configurable": {"thread_id": thread_id}}
        state      = graph.get_state(config)
        spider_data = state.values.get("player_output")

        if not spider_data:
            print(f"❌ No spider data found for thread: {thread_id}")
            print("   Run a player_analysis query first.")
            return

        # ── Step 2: Extract data ──
        player_name = spider_data.get("player_name", "Unknown")
        position    = spider_data.get("position", "—")
        categories  = spider_data.get("categories", [])
        scores      = spider_data.get("scores", [])
        fifa        = spider_data.get("fifa_stats", {})
        med         = spider_data.get("medical_report", {})

        # ── Step 3: Safe copies — no mutation ──
        categories_plot = categories + [categories[0]]
        scores_plot     = scores     + [scores[0]]

        # ── Step 4: Color per score ──
        def score_color(s):
            if s >= 85: return "#00FF88"
            if s >= 70: return "#FFD700"
            if s >= 55: return "#FF8C00"
            return "#FF4444"

        overall      = round(sum(scores) / len(scores), 1)
        status       = med.get("Status", "—")
        health       = med.get("Health_Score", 0)
        sleep        = med.get("Sleep_Quality", 0)
        soreness     = med.get("Soreness_Level", 0)
        status_color = score_color(health)

        # ── Step 5: Build figure with 2 subplots ──
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "polar"}, {"type": "table"}]],
            column_widths=[0.6, 0.4],
        )

        # ── Spider chart ──
        fig.add_trace(
            go.Scatterpolar(
                r           = scores_plot,
                theta       = categories_plot,
                fill        = 'toself',
                fillcolor   = 'rgba(0, 255, 136, 0.25)',
                line        = dict(color='#00FF88', width=3),
                marker      = dict(
                    size  = 8,
                    color = [score_color(s) for s in scores_plot],
                    line  = dict(color='white', width=1)
                ),
                name        = player_name,
                hovertemplate = "<b>%{theta}</b><br>Score: %{r}<extra></extra>",
            ),
            row=1, col=1
        )

        # ── Stats table ──
        fig.add_trace(
            go.Table(
                header=dict(
                    values    = ["<b>Attribute</b>", "<b>Value</b>"],
                    fill_color = "#0d0d2e",
                    font      = dict(color="white", size=13),
                    align     = "center",
                    height    = 32,
                ),
                cells=dict(
                    values = [
                        [
                            "🏃 Position",
                            "⭐ Overall",
                            "⚡ Pace (PAC)",
                            "🏃 Stamina (STA)",
                            "💪 Physical (PHY)",
                            "🏥 Health Score",
                            "😴 Sleep Quality",
                            "🔥 Soreness",
                            "📋 Status",
                        ],
                        [
                            position,
                            f"{overall}/100",
                            f"{fifa.get('PAC', '—')}/99",
                            f"{fifa.get('STA', '—')}/99",
                            f"{fifa.get('PHY', '—')}/99",
                            f"{health}/100",
                            f"{sleep}/10",
                            f"{soreness}/10",
                            status,
                        ]
                    ],
                    fill_color = [
                        ["#0d0d1a"] * 9,
                        [
                            "#0d0d1a",
                            "#0d0d1a",
                            score_color(fifa.get('PAC', 0)),
                            score_color(fifa.get('STA', 0)),
                            score_color(fifa.get('PHY', 0)),
                            score_color(health),
                            "#0d0d1a",
                            "#0d0d1a",
                            status_color,
                        ]
                    ],
                    font      = dict(
                        color = ["#aaaaaa", "white"],
                        size  = 12
                    ),
                    align     = ["left", "center"],
                    height    = 30,
                )
            ),
            row=1, col=2
        )

        # ── Layout ──
        fig.update_layout(
            title = dict(
                text = (
                    f"<b>{player_name.upper()}</b>"
                    f"  ·  {position}"
                    f"  ·  Overall: <span style='color:#FFD700'>{overall}</span>"
                    f"  ·  <span style='color:{status_color}'>{status}</span>"
                ),
                font = dict(size=20, color="white"),
                x    = 0.5,
            ),
            polar = dict(
                radialaxis = dict(
                    visible   = True,
                    range     = [0, 100],
                    gridcolor = "rgba(255,255,255,0.1)",
                    tickfont  = dict(color="rgba(255,255,255,0.5)", size=10),
                    tickvals  = [20, 40, 60, 80, 100],
                ),
                angularaxis = dict(
                    gridcolor = "rgba(255,255,255,0.1)",
                    tickfont  = dict(color="white", size=13, family="Segoe UI"),
                ),
                bgcolor = "rgb(13,13,30)",
            ),
            paper_bgcolor = "rgb(13,13,26)",
            plot_bgcolor  = "rgb(13,13,26)",
            showlegend    = False,
            height        = 500,
            margin        = dict(t=80, b=20, l=20, r=20),
        )

        # ── Print summary ──
        print(f"\n{'━'*60}")
        print(f"📊 SPIDER DIAGRAM — {player_name.upper()}")
        print(f"{'━'*60}")
        print(f"   Position  : {position}")
        print(f"   Overall   : {overall}/100")
        print(f"   Status    : {status}")
        print(f"{'─'*60}")
        for cat, sc in zip(categories, scores):
            bar   = "█" * int(sc / 5)
            space = "░" * (20 - int(sc / 5))
            color_label = (
                "🟢" if sc >= 85 else
                "🟡" if sc >= 70 else
                "🟠" if sc >= 55 else "🔴"
            )
            print(f"   {color_label} {cat:<12} │{bar}{space}│ {sc}")
        print(f"{'━'*60}\n")

        fig.show()

    except Exception as e:
        print(f"⚠️ Error rendering chart: {e}")
        import traceback
        traceback.print_exc()


# ── Run with current thread ──
print(f"📊 Rendering spider diagram for thread: {CURRENT_THREAD}")
display_spider_diagram_from_memory(CURRENT_THREAD)

📊 Rendering spider diagram for thread: session-210311423116

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 SPIDER DIAGRAM — BILL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Position  : Outside Back
   Overall   : 84.0/100
   Status    : Fit to Play
────────────────────────────────────────────────────────────
   🟢 Pace         │███████████████████░│ 98
   🟠 Stamina      │█████████████░░░░░░░│ 66
   🟢 Physicality  │█████████████████░░░│ 89
   🟢 Health       │█████████████████░░░│ 87
   🟡 Technical    │████████████████░░░░│ 80.0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

